# 🚀 Extrator Histórico PT com Ollama no Google Colab (Gratuito)

Este notebook permite executar a recolha histórica de notícias e efemérides Wikipedia para o **repositório Português (PT)** de forma automática e sem limites, utilizando o **Ollama** de forma **100% gratuita** aproveitando as GPUs do Google Colab.

--- 
### ⚠️ PASSO CRÍTICO: Ativar GPU
Antes de começar, garante que o Colab está a usar uma GPU para o Ollama ser extremamente rápido:
1. No menu superior, vai a **Runtime** (Ambiente de Execução) -> **Change runtime type** (Alterar tipo de ambiente de execução).
2. Em *Hardware accelerator* (Acelerador de Hardware), seleciona **T4 GPU** (ou outra GPU disponível).
3. Clica em **Save**.

## 1. Instalar o Ollama no Colab

In [ ]:
# Instala o zstd e pciutils (necessário para a correta instalação e deteção de GPU do Ollama)
!sudo apt-get update && sudo apt-get install -y zstd pciutils

# Descarrega e instala o binário oficial do Ollama para Linux
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Iniciar o Servidor Ollama em Background

In [ ]:
import subprocess
import time
import requests

print("A fechar processos Ollama anteriores (se existirem)...")
subprocess.run(["pkill", "-f", "ollama"])
time.sleep(2)

print("A iniciar o servidor Ollama em segundo plano...")
# Gravar logs em /content/ollama.log para diagnóstico rápido em caso de falha
log_file = open("/content/ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

# Aguarda 6 segundos para o servidor iniciar
time.sleep(6)

# Testa a ligação local
try:
    res = requests.get("http://localhost:11434")
    if res.status_code == 200:
        print("Ollama iniciado com sucesso e a responder! ✔️")
    else:
        print(f"Ollama iniciou mas respondeu com status {res.status_code}.")
except Exception as e:
    print(f"❌ Erro ao ligar ao Ollama: {e}")
    print("\n--- Últimas linhas do log do Ollama (/content/ollama.log) ---")
    try:
        with open("/content/ollama.log", "r") as f:
            lines = f.readlines()
            for line in lines[-20:]:
                print(line, end="")
    except Exception:
        pass

## 3. Descarregar o Modelo de Inteligência Artificial
Utilizamos o **gemma4:e2b** (modelo leve e eficiente de 2B parâmetros), otimizado para alto desempenho e sem riscos de travamento em ambientes Colab.

In [ ]:
# Descarrega o modelo gemma4:e2b
!ollama pull gemma4:e2b

## 4. Conectar ao Google Drive
Corre esta célula para permitir que o Colab leia e grave as notícias diretamente nas pastas do teu Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Instalar dependências dos Scrapers

In [ ]:
!pip install requests==2.34.2 beautifulsoup4==4.15.0 instaloader==4.15.2 playwright==1.61.0
!playwright install

## 6. Execução Automática (Sem Limites)
Esta célula vai processar todos os anos planeados (ex: 2005 a 2026) **mês a mês** de forma sequencial e sem limites para a língua portuguesa.
Nota: Ajusta `start_year` se quiseres recomeçar noutro ano.

In [ ]:
import os
import sys
import subprocess

# Configuração da recolha (começa no progresso atual: 2005)
start_year = 2005
end_year = 2026
model = "gemma4:e2b"

# Caminho absoluto do repositório no teu Google Drive
path_pt = "/content/drive/MyDrive/NoticiasOntemProject/Notícias de Ontem"

def run_command_streaming(cmd):
    # Inicia o processo em background capturando stdout e stderr juntos
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=0)
    
    # Lê caractere a caractere para suportar os carriage returns (\r) de progresso em tempo real no Colab
    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        if char:
            sys.stdout.write(char)
            sys.stdout.flush()
            
    return process.poll()

print(f"📢 A iniciar extração PT de {start_year} a {end_year} usando {model}...")

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        print(f"\n" + "=" * 60)
        print(f"👉 A PROCESSAR PT: Ano {year} - Mês {month:02d}")
        print("=" * 60)
        
        if os.path.exists(path_pt):
            os.chdir(path_pt)
            cmd_pt = f"python -u run_historical_scrapers.py --start-year {year} --end-year {year} --months {month} --limit 0 --lang pt --local-llm {model}"
            run_command_streaming(cmd_pt)
        else:
            print(f"⚠ Caminho PT não encontrado: {path_pt}")

print("\n🎉 EXTRAÇÃO HISTÓRICA PT CONCLUÍDA!")